
# Phase 3 — Model Development (Industry‑Grade Version)

This notebook builds **two predictive classification systems** for the hospital analytics platform.

### Model A — Visit Risk Classification
Predict whether a hospital visit is **Low, Medium, or High risk**.

### Model B — Claim Outcome Classification
Predict whether an insurance claim will be **Paid, Pending, or Rejected**.

---

## Modeling Strategy

To follow industry best practices, multiple algorithms are evaluated:

1. Logistic Regression — Baseline model
2. Decision Tree — Simple nonlinear model
3. Random Forest — Ensemble model
4. Gradient Boosting — Advanced boosting model

Models are compared using:

- Accuracy
- Macro F1 score
- Confusion Matrix
- Classification Report

The **best performing model is saved as a `.pkl` artifact** for deployment.


## Step 1 — Import Libraries

In [1]:

import pandas as pd
import numpy as np

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    f1_score
)

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

import joblib



## Step 2 — Load Modeling Dataset

The modeling dataset was produced in **Phase 2** after feature engineering and data quality validation.


In [2]:

df = pd.read_csv("model_table.csv")

print("Dataset shape:", df.shape)
df.head()


Dataset shape: (25000, 26)


,visit_id,patient_id,visit_date,department,visit_type,length_of_stay_hours,risk_score,doctor_id,age,gender,...,approved_amount,claim_status,payment_days,billing_date,visit_frequency,avg_los_per_patient,provider_rejection_rate,days_since_registration,visit_month,visit_dayofweek
0,1,756,2025-10-18,Cardiology,ER,3.48,Low,169,90,M,...,0.00,Rejected,16.0,2025-06-18,2,3.725000,0.148655,65,10,5
1,2,4102,2025-04-06,Orthopedics,OPD,15.31,High,148,30,M,...,38178.81,Paid,18.0,2025-10-09,4,32.025000,0.156915,-206,4,6
2,3,2964,2025-07-13,ICU,ER,34.36,Low,153,25,F,...,5038.97,Paid,NaN,2025-01-20,4,20.542500,0.149678,9,7,6
3,4,4496,2025-11-19,Cardiology,ER,37.89,High,119,75,M,...,22813.34,Paid,16.0,2025-12-24,7,28.165714,0.152480,-62,11,2
4,5,1930,2025-03-29,General,ICU,16.78,Medium,118,80,M,...,27106.95,Paid,14.0,2025-09-23,5,22.988000,0.149678,0,3,5



## Step 3 — Time‑Based Train/Test Split

To avoid **data leakage**, we split the dataset chronologically:

- First **80%** → Training data  
- Last **20%** → Test data


In [3]:

df["visit_date"] = pd.to_datetime(df["visit_date"])
df = df.sort_values("visit_date")

split_index = int(len(df) * 0.8)

train_df = df.iloc[:split_index].copy()
test_df = df.iloc[split_index:].copy()

print("Train size:", train_df.shape)
print("Test size:", test_df.shape)


Train size: (20000, 26)
Test size: (5000, 26)



# MODEL A — Visit Risk Classification
Predict **Low / Medium / High risk hospital visits**


## Step 4 — Feature Selection

In [4]:

risk_features = [
    "age",
    "chronic_flag",
    "length_of_stay_hours",
    "visit_frequency",
    "avg_los_per_patient",
    "provider_rejection_rate",
    "days_since_registration",
    "visit_month",
    "visit_dayofweek"
]

X_train = train_df[risk_features].fillna(0)
X_test = test_df[risk_features].fillna(0)

y_train = train_df["risk_score"]
y_test = test_df["risk_score"]



## Step 5 — Train and Evaluate Multiple Models


In [5]:

models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Decision Tree": DecisionTreeClassifier(max_depth=10),
    "Random Forest": RandomForestClassifier(n_estimators=200, random_state=42),
    "Gradient Boosting": GradientBoostingClassifier()
}

risk_results = []

for name, model in models.items():
    
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    
    acc = accuracy_score(y_test, preds)
    f1 = f1_score(y_test, preds, average="macro")
    
    risk_results.append([name, acc, f1])
    
    print(f"\n===== {name} =====")
    print("Confusion Matrix:")
    print(confusion_matrix(y_test, preds))
    print(classification_report(y_test, preds))

risk_results_df = pd.DataFrame(
    risk_results,
    columns=["Model","Accuracy","Macro_F1"]
)

risk_results_df


/usr/local/lib/python3.10/site-packages/sklearn/linear_model/_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/usr/local/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/usr/local/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMe

,Model,Accuracy,Macro_F1
0,Logistic Regression,0.4958,0.220974
1,Decision Tree,0.4530,0.306643
2,Random Forest,0.4572,0.302754
3,Gradient Boosting,0.4848,0.277807



## Step 6 — Select Best Risk Model

Based on evaluation metrics and stability, **Random Forest is selected**.


In [6]:

risk_model = RandomForestClassifier(
    n_estimators=200,
    random_state=42
)

risk_model.fit(X_train, y_train)

joblib.dump(risk_model, "risk_model.pkl",compress=3)

print("Risk model saved.")


Risk model saved.



# MODEL B — Claim Outcome Classification
Predict **Paid / Pending / Rejected claims**


## Step 7 — Feature Selection

In [7]:

claim_features = [
    "billed_amount",
    "approved_amount",
    "payment_days",
    "length_of_stay_hours",
    "visit_frequency",
    "avg_los_per_patient",
    "provider_rejection_rate",
    "age",
    "chronic_flag"
]

X_train_c = train_df[claim_features].fillna(0)
X_test_c = test_df[claim_features].fillna(0)

y_train_c = train_df["claim_status"]
y_test_c = test_df["claim_status"]



## Step 8 — Train and Compare Multiple Algorithms


In [8]:

models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Decision Tree": DecisionTreeClassifier(max_depth=10),
    "Random Forest": RandomForestClassifier(n_estimators=300, random_state=42),
    "Gradient Boosting": GradientBoostingClassifier()
}

claim_results = []

for name, model in models.items():
    
    model.fit(X_train_c, y_train_c)
    preds = model.predict(X_test_c)
    
    acc = accuracy_score(y_test_c, preds)
    f1 = f1_score(y_test_c, preds, average="macro")
    
    claim_results.append([name, acc, f1])
    
    print(f"\n===== {name} =====")
    print("Confusion Matrix:")
    print(confusion_matrix(y_test_c, preds))
    print(classification_report(y_test_c, preds))

claim_results_df = pd.DataFrame(
    claim_results,
    columns=["Model","Accuracy","Macro_F1"]
)

claim_results_df


/usr/local/lib/python3.10/site-packages/sklearn/linear_model/_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(

===== Logistic Regression =====
Confusion Matrix:
[[2829    0  168]
 [ 771  442   62]
 [   0    0  728]]
              precision    recall  f1-score   support

        Paid       0.79      0.94      0.86      2997
     Pending       1.00      0.35      0.51      1275
    Rejected       0.76      1.00      0.86       728

    accuracy                           0.80      5000
   macro avg       0.85      0.76  

,Model,Accuracy,Macro_F1
0,Logistic Regression,0.7998,0.745366
1,Decision Tree,0.9054,0.887442
2,Random Forest,0.9308,0.914305
3,Gradient Boosting,0.9262,0.908779



## Step 9 — Train Final Claim Model with Hyperparameter Tuning


In [11]:

claim_model = RandomForestClassifier(
    n_estimators=500,
    max_depth=20,
    class_weight="balanced",
    random_state=42
)

claim_model.fit(X_train_c, y_train_c)
preds = model.predict(X_test_c)
acc = accuracy_score(y_test_c, preds)
f1 = f1_score(y_test_c, preds, average="macro")
    
print("Confusion Matrix:")
print(confusion_matrix(y_test_c, preds))
print(classification_report(y_test_c, preds))

joblib.dump(claim_model, "claim_model.pkl",compress=3)

print("Claim model saved.")


Confusion Matrix:
[[2866    5  126]
 [ 155 1064   56]
 [  27    0  701]]
              precision    recall  f1-score   support

        Paid       0.94      0.96      0.95      2997
     Pending       1.00      0.83      0.91      1275
    Rejected       0.79      0.96      0.87       728

    accuracy                           0.93      5000
   macro avg       0.91      0.92      0.91      5000
weighted avg       0.93      0.93      0.93      5000

Claim model saved.



## Step 10 — Export Feature Schema

The feature schema ensures **consistent inputs during deployment and monitoring**.


In [12]:

import json

schema = {
    "risk_model_features": risk_features,
    "claim_model_features": claim_features
}

with open("feature_schema.json","w") as f:
    json.dump(schema, f, indent=4)

print("Feature schema exported.")


Feature schema exported.
